# Social Network BO Demo

Minimal Bayesian optimization on the Enron graph with `GRF + Thompson`, `BFS`, `DFS`, and `RandomSearch`.

In [1]:
import os
import sys
from collections import deque
from pathlib import Path

cwd = Path.cwd().resolve()
project_root = next(path for path in [cwd, *cwd.parents] if (path / "data").exists() and (path / "experiments").exists())
venv_site_packages = str(project_root / "venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages")
demo_dir = os.path.join(project_root, "experiments", "bayesopt", "social_networks")
project_root = str(project_root)
if venv_site_packages not in sys.path:
    sys.path.append(venv_site_packages)
if demo_dir not in sys.path:
    sys.path.append(demo_dir)
if project_root not in sys.path:
    sys.path.append(project_root)

import gpytorch
import numpy as np
import torch
from tqdm.auto import tqdm
from linear_operator import settings
from gpytorch import settings as gsettings

from grf_gp.sampler import GRFSampler
from grf_gp.kernels.general import GeneralGRFKernel
from grf_gp.model import GRFGP
from grf_gp.utils.config import set_gp_defaults

from data_utils import prepare_social_network_data
SEED = 123
DATASET_NAME = "enron"
N_BO_STEPS = 20
BO_BATCH_SIZE = 10
MAX_WALK_LENGTH = 3
WALKS_PER_NODE = 1000
P_HALT = 0.5
TRAIN_LR = 0.01
TRAIN_ITERS = 100
GP_RETRAIN_INTERVAL = 5 * BO_BATCH_SIZE

# set_gp_defaults(linear_operator_settings=settings, gpytorch_settings=gsettings)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
data = prepare_social_network_data(DATASET_NAME, dtype=np.float64)

X = data["X"]
Y = data["y"]
Y_norm = (Y - Y.mean()) / Y.std()
adjacency = data["adjacency"]
GROUND_TRUTH_BEST = float(Y.max())
num_nodes = data["num_nodes"]

neighbors = [adjacency.indices[adjacency.indptr[i]:adjacency.indptr[i + 1]] for i in range(num_nodes)]

adjacency_torch = torch.sparse_csr_tensor(
    torch.tensor(adjacency.indptr, dtype=torch.int64),
    torch.tensor(adjacency.indices, dtype=torch.int64),
    torch.tensor(adjacency.data, dtype=torch.float64),
    size=adjacency.shape,
)

sampler = GRFSampler(
    adjacency_torch,
    walks_per_node=WALKS_PER_NODE,
    p_halt=P_HALT,
    max_walk_length=MAX_WALK_LENGTH,
    seed=SEED,
    use_tqdm=True,
)
rw_mats = sampler()

print(f"dataset: {DATASET_NAME}")
print(f"nodes: {data['num_nodes']:,}")
print(f"edges: {data['num_edges']:,}")
print(f"degree mean/std: {data['y_mean']:.4f} / {data['y_std']:.4f}")
print(f"ground-truth best: {GROUND_TRUTH_BEST:.1f}")


/var/folders/0v/vn4885796ql7_mlpq_dc1n9r0000gn/T/ipykernel_90952/4236278519.py:12: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  adjacency_torch = torch.sparse_csr_tensor(


dataset: enron
nodes: 36,692
edges: 183,831
degree mean/std: 10.0202 / 36.1000
ground-truth best: 1383.0


In [3]:
def train_model(
    model,
    likelihood,
    x_train,
    y_train,
    lr=0.01,
    max_iter=100,
    print_every=None,
    progress_desc="Training",
):
    model.train()
    likelihood.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    pbar = tqdm(total=max_iter, desc=progress_desc, leave=False)
    for step in range(max_iter):
        optimizer.zero_grad()
        train_loss = -mll(model(x_train), y_train)
        train_loss.backward()
        optimizer.step()

        loss_value = train_loss.item()
        if print_every and ((step + 1) % print_every == 0 or (step + 1) == max_iter):
            print(f"Iter {step + 1}/{max_iter} - Loss: {loss_value:.4f}")
        pbar.set_postfix({"loss": f"{loss_value:.4f}"})
        pbar.update(1)

    pbar.close()


def unobserved_nodes(observed_set):
    observed = np.fromiter(observed_set, dtype=np.int64)
    return np.setdiff1d(X, observed, assume_unique=False)


def random_search(observed, observed_set, state, rng):
    pool = unobserved_nodes(observed_set)
    batch_size = min(BO_BATCH_SIZE, len(pool))
    return rng.choice(pool, size=batch_size, replace=False).tolist()


def bfs(observed, observed_set, state, rng):
    batch = []
    frontier = state["bfs_frontier"]
    while frontier and len(batch) < BO_BATCH_SIZE:
        node = frontier.popleft()
        if node not in observed_set:
            batch.append(node)
    if len(batch) < BO_BATCH_SIZE:
        extra = random_search(observed, observed_set.union(batch), state, rng)
        batch.extend(extra[: BO_BATCH_SIZE - len(batch)])
    return batch


def dfs(observed, observed_set, state, rng):
    batch = []
    frontier = state["dfs_frontier"]
    while frontier and len(batch) < BO_BATCH_SIZE:
        node = frontier.pop()
        if node not in observed_set:
            batch.append(node)
    if len(batch) < BO_BATCH_SIZE:
        extra = random_search(observed, observed_set.union(batch), state, rng)
        batch.extend(extra[: BO_BATCH_SIZE - len(batch)])
    return batch


def extend_frontier(frontier, nodes):
    for node in nodes:
        frontier.extend(int(neighbor) for neighbor in neighbors[node])


class GRFThompson:
    def __init__(self, retrain_interval):
        self.retrain_interval = retrain_interval
        self.model = None
        self.last_fit_size = 0

    def fit(self, observed):
        x_train = torch.tensor(observed, dtype=torch.long, device=device)
        y_train = torch.tensor(Y_norm[observed], dtype=torch.float64, device=device)
        likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device=device, dtype=torch.float64)
        kernel = GeneralGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device=device, dtype=torch.float64)
        self.model = GRFGP(x_train, y_train, likelihood, kernel).to(device=device, dtype=torch.float64)
        train_model(
            self.model,
            likelihood,
            x_train,
            y_train,
            lr=TRAIN_LR,
            max_iter=TRAIN_ITERS,
            progress_desc="GRF fit",
        )
        self.model.eval()
        self.last_fit_size = len(observed)

    def __call__(self, observed, observed_set, state, rng):
        pool = unobserved_nodes(observed_set)
        batch_size = min(BO_BATCH_SIZE, len(pool))
        if len(observed) == 0:
            return rng.choice(pool, size=batch_size, replace=False).tolist()

        needs_refit = self.model is None or (len(observed) - self.last_fit_size) >= self.retrain_interval
        if needs_refit:
            self.fit(observed)

        x_pool = torch.tensor(pool, dtype=torch.long, device=device)
        with torch.no_grad():
            sample = self.model.predict_sample(x_pool, n_samples=1).squeeze(0)
        order = torch.argsort(sample, descending=True)
        return x_pool[order[:batch_size]].cpu().tolist()


def run_bo(name, select_batch, seed):
    rng = np.random.default_rng(seed)
    observed = []
    observed_set = set()
    best_values = []
    state = {"bfs_frontier": deque(), "dfs_frontier": []}

    for step in range(N_BO_STEPS):
        batch = select_batch(observed, observed_set, state, rng)
        observed.extend(batch)
        observed_set.update(batch)
        extend_frontier(state["bfs_frontier"], batch)
        extend_frontier(state["dfs_frontier"], batch)

        best = float(Y[observed].max())
        best_values.append(best)
        print(f"{name:15s} step {step + 1:02d} | batch={len(batch):3d} | visited={len(observed):4d} | best={best:8.1f} | gt_best={GROUND_TRUTH_BEST:8.1f}")

    return observed, best_values



In [ ]:
results = {}

algorithms = {
    "GRF+Thompson": GRFThompson(GP_RETRAIN_INTERVAL),
    "BFS": bfs,
    "DFS": dfs,
    "RandomSearch": random_search,
}

for run_id, (name, select_batch) in enumerate(algorithms.items()):
    print(f"\n=== {name} ===")
    observed_nodes, best_values = run_bo(name, select_batch, seed=SEED + run_id)
    results[name] = {
        "observed_nodes": observed_nodes,
        "best_values": best_values,
    }



=== GRF+Thompson ===
GRF+Thompson    step 01 | batch= 10 | visited=  10 | best=   924.0 | gt_best=  1383.0


GRF fit:   0%|          | 0/100 [00:00<?, ?it/s]

/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/linear_operator/utils/sparse.py:51: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  if nonzero_indices.storage():
/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/linear_operator/utils/sparse.py:66: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:656.)
  res = cls(index_tensor, value_tensor, interp_size)


GRF+Thompson    step 02 | batch= 10 | visited=  20 | best=  1099.0 | gt_best=  1383.0
GRF+Thompson    step 03 | batch= 10 | visited=  30 | best=  1099.0 | gt_best=  1383.0
GRF+Thompson    step 04 | batch= 10 | visited=  40 | best=  1367.0 | gt_best=  1383.0
GRF+Thompson    step 05 | batch= 10 | visited=  50 | best=  1367.0 | gt_best=  1383.0
GRF+Thompson    step 06 | batch= 10 | visited=  60 | best=  1367.0 | gt_best=  1383.0


GRF fit:   0%|          | 0/100 [00:00<?, ?it/s]

GRF+Thompson    step 07 | batch= 10 | visited=  70 | best=  1367.0 | gt_best=  1383.0
GRF+Thompson    step 08 | batch= 10 | visited=  80 | best=  1367.0 | gt_best=  1383.0
GRF+Thompson    step 09 | batch= 10 | visited=  90 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 10 | batch= 10 | visited= 100 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 11 | batch= 10 | visited= 110 | best=  1383.0 | gt_best=  1383.0


GRF fit:   0%|          | 0/100 [00:00<?, ?it/s]

GRF+Thompson    step 12 | batch= 10 | visited= 120 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 13 | batch= 10 | visited= 130 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 14 | batch= 10 | visited= 140 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 15 | batch= 10 | visited= 150 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 16 | batch= 10 | visited= 160 | best=  1383.0 | gt_best=  1383.0


GRF fit:   0%|          | 0/100 [00:00<?, ?it/s]

GRF+Thompson    step 17 | batch= 10 | visited= 170 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 18 | batch= 10 | visited= 180 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 19 | batch= 10 | visited= 190 | best=  1383.0 | gt_best=  1383.0
GRF+Thompson    step 20 | batch= 10 | visited= 200 | best=  1383.0 | gt_best=  1383.0

=== BFS ===
BFS             step 01 | batch= 10 | visited=  10 | best=    32.0 | gt_best=  1383.0
BFS             step 02 | batch= 10 | visited=  20 | best=   505.0 | gt_best=  1383.0
BFS             step 03 | batch= 10 | visited=  30 | best=   505.0 | gt_best=  1383.0
BFS             step 04 | batch= 10 | visited=  40 | best=   505.0 | gt_best=  1383.0
BFS             step 05 | batch= 10 | visited=  50 | best=  1245.0 | gt_best=  1383.0
BFS             step 06 | batch= 10 | visited=  60 | best=  1245.0 | gt_best=  1383.0
BFS             step 07 | batch= 10 | visited=  70 | best=  1383.0 | gt_best=  1383.0
BFS             step 08 | batch= 10 | vis